# What people type when they can't name a country

A quick tour of the [MapGuesser](https://mapguesser.fr/en) answers dataset: in this geography game a country lights up on a world map and players **type** its name. Every wrong answer tells us what they named instead.

Full write-up: [Hawaii, Dubai, Bali: what people type when they can't name the country](https://mapguesser.fr/en/blogs/hawaii-dubai-bali-what-people-type-when-they-cant-name-the-country).

In [ ]:
import glob, os
import pandas as pd
import matplotlib.pyplot as plt

base = os.path.dirname(glob.glob('/kaggle/input/**/country_difficulty.csv', recursive=True)[0])
difficulty = pd.read_csv(f'{base}/country_difficulty.csv')
pairs = pd.read_csv(f'{base}/confusion_pairs.csv')
places = pd.read_csv(f'{base}/typed_places.csv')
countries = pd.read_csv(f'{base}/countries.csv')
print(f'{difficulty.rounds.sum():,} rounds, {len(pairs):,} confusion pairs, {places.typed_place.nunique()} non-country places')

## 1. The hardest countries to name

Fail rate = rounds where the player never found the answer. Only UN members with at least 500 rounds, to keep remote territories out.

In [ ]:
un = countries.loc[countries.un_member == 1, 'code']
hard = (difficulty[difficulty.code.isin(un) & (difficulty.rounds >= 500)]
        .nlargest(15, 'fail_rate_pct')
        .sort_values('fail_rate_pct'))
ax = hard.plot.barh(x='name_en', y='fail_rate_pct', legend=False, figsize=(8, 6), color='#b8471f')
ax.set_xlabel('rounds failed (%)'); ax.set_ylabel('')
ax.set_title('Hardest UN countries to name on a map')
plt.tight_layout(); plt.show()

## 2. The pairs people mix up

`pct_of_shown_rounds`: share of the rounds of the shown country where the player typed the other one.

In [ ]:
pairs[pairs.rounds >= 100].nlargest(15, 'pct_of_shown_rounds')[['shown_name', 'typed_name', 'rounds', 'pct_of_shown_rounds']]

Some confusions are lopsided: San Marino got "Vatican City" 1,180 times, Vatican City got "San Marino" only 23 times. The `ratio` column compares both directions.

In [ ]:
both = pairs.merge(pairs, left_on=['shown_code', 'typed_code'], right_on=['typed_code', 'shown_code'], suffixes=('', '_rev'))
both = both[both.shown_code < both.typed_code]
both.assign(ratio=(both.answers / both.answers_rev).round(1)).nlargest(10, 'answers')[['shown_name', 'typed_name', 'answers', 'answers_rev', 'ratio']]

## 3. Places that are not countries

When the name is missing, players type the famous place they know nearby: Hawaii for the Pacific, Dubai for the Gulf, Sicily for Malta.

In [ ]:
top = places.groupby('typed_place').answers.sum().nlargest(12).sort_values()
ax = top.plot.barh(figsize=(8, 5), color='#2f6f7d')
ax.set_xlabel('wrong answers'); ax.set_ylabel('')
ax.set_title('Non-country places typed instead of a country')
plt.tight_layout(); plt.show()

In [ ]:
places[places.typed_place == 'Hawaii'].nlargest(10, 'answers')[['shown_name', 'answers', 'players']]

## Caveats

- Answers are **typed**, not chosen: this measures recall, not recognition.
- Players chose to play a geography game; the audience leans French (26% of players with a known country).
- Micro-states are tiny on screen.

Data under CC BY 4.0: please credit MapGuesser with a link to https://mapguesser.fr/en. Scripts: https://github.com/CallMeAl3x/mapguesser-geography-answers